In [8]:
# 模拟数据
import pandas as pd

data = {
    "id": [1,2,3,4,5,6,7],
    "runway": ["05L","05L","05L","05L","05L","23R","23R"],
    "device_id": ["RVR_A","RVR_A","RVR_A","RVR_B","RVR_B","RVR_C","RVR_C"],
    "obs_time": [
        "2026-06-29 08:00:00",
        "2026-06-29 08:01:00",
        "2026-06-29 08:02:00",
        "2026-06-29 08:00:00",
        "2026-06-29 08:01:00",
        "2026-06-29 08:00:00",
        "2026-06-29 08:01:00"
    ],
    "rvr_value": [800,820,None,760,0,1200,None],
    "status": ["OK","OK","MISSING","OK","ERROR","OK","MISSING"]
}

df = pd.DataFrame(data)

df


,id,runway,device_id,obs_time,rvr_value,status
0,1,05L,RVR_A,2026-06-29 08:00:00,800.0,OK
1,2,05L,RVR_A,2026-06-29 08:01:00,820.0,OK
2,3,05L,RVR_A,2026-06-29 08:02:00,NaN,MISSING
3,4,05L,RVR_B,2026-06-29 08:00:00,760.0,OK
4,5,05L,RVR_B,2026-06-29 08:01:00,0.0,ERROR
5,6,23R,RVR_C,2026-06-29 08:00:00,1200.0,OK
6,7,23R,RVR_C,2026-06-29 08:01:00,NaN,MISSING


## 题目: 数据清洗 + 分组统计

* **要求：**

    * **分别用 SQL 和 Pandas 完成：**

        - 查询每个 device_id 的总记录数。
        - 查询每个 device_id 的缺失记录数：rvr_value IS NULL。
        - 查询每个 device_id 的异常记录数：rvr_value = 0 或 status = 'ERROR'。
        - 计算每个设备的有效数据均值：只统计 rvr_value > 0 且不为空的数据。
    * **输出字段：**

| device_id | total_count | missing_count | error_count | valid_avg_rvr |

In [ ]:
import duckdb
query = """
SELECT  
    device_id,
    COUNT(*)::INTEGER AS total_count,

    SUM(
        CASE WHEN rvr_value IS NULL THEN 1 ELSE 0 END
    ) ::INTEGER AS missing_count,

    SUM(
        CASE WHEN rvr_value = 0 OR status = 'ERROR' THEN 1 ELSE 0 END
    ) ::INTEGER AS error_count,

    AVG(
        CASE WHEN rvr_value >0 AND rvr_value IS NOT NULL THEN rvr_value END
    ) ::INTEGER AS valid_avg_rvr
FROM df
GROUP BY device_id
"""
df_sql = duckdb.execute(query).fetchdf()
print(df_sql)

In [16]:
# pandas轨道
df['is_missing'] = df['rvr_value'].isna()

df['is_abnormal'] = (df['rvr_value'] == 0) | (df['status'] == 'ERROR')

df['valid_num'] = df['rvr_value'].where((df['rvr_value'] > 0) & (df['rvr_value'].notna()))

df_pandas = (
    df
    .groupby('device_id').agg(
        total_count = ('id','size'),
        missing_count = ('is_missing','sum'),
        error_count = ('is_abnormal','sum'),
        valid_avg_rvr = ('valid_num','mean')
    )
    .reset_index()
)
print(df_pandas)

  device_id  total_count  missing_count  error_count  valid_avg_rvr
0     RVR_A            3              1            0          810.0
1     RVR_B            2              0            1          760.0
2     RVR_C            2              1            0         1200.0
